<a href="https://colab.research.google.com/github/abualjoud95/colab_repo/blob/main/Qwen3_5_(2B)_FineTune_Classification_Results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CRM AI Ticket Intelligence — Model Comparison

This notebook compares:
- **Original:** the base Qwen3.5 model.
- **Fine-tuned:** the same base model with your saved LoRA adapter.

It runs the models **sequentially** to avoid loading both into GPU memory at once.

## Before running
1. On your Mac, zip the `crm-ticket-lora` folder. The folder should contain `adapter_config.json` and `adapter_model.safetensors` (or equivalent adapter weights).
2. Upload the ZIP to Google Drive, for example: `MyDrive/CRM_AI/crm-ticket-lora.zip`.
3. Use a Colab GPU runtime.
4. Make sure the `SYSTEM_PROMPT` below matches the prompt used during training.

**Dataset note:** the test split is recreated from the published dataset using the same cleaning and `random_state=42` procedure previously proposed. This is reproducible only if the training notebook used that exact dataset revision and exact cleaning/splitting code. If it did not, the recreated test set may differ from the original training test set.


In [1]:
# Install only if these packages are not already available in this runtime.
# Prefer matching the versions used by your training notebook.
%pip install -q unsloth peft datasets scikit-learn


## 1. Check the runtime and install dependencies

Use the same compatible Unsloth / Transformers environment as the training notebook wherever possible. If you encounter version conflicts, stop and align the package versions with the training environment rather than upgrading packages randomly.

In [2]:
import torch, gc, os, json, time, re
import pandas as pd
from pathlib import Path

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Runtime > Change runtime type.")

print("GPU:", torch.cuda.get_device_name(0))
print("Total GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
Total GPU memory (GB): 14.56


## 2. Mount Google Drive and locate the LoRA adapter

On your Mac, create a ZIP of the adapter folder and upload it to Google Drive.

Example Mac Terminal command (change the path to your actual folder):
```bash
cd "/path/to/the/parent/folder"
zip -r crm-ticket-lora.zip crm-ticket-lora
```

The ZIP should contain the adapter files. Do not upload only a shortcut to the folder.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

ADAPTER_ZIP = "/content/drive/MyDrive/CRM_AI/crm-ticket-lora.zip"
WORK_DIR = "/content/crm_ai_comparison"
ADAPTER_DIR = f"{WORK_DIR}/crm-ticket-lora"
RESULTS_DIR = "/content/drive/MyDrive/CRM_AI/comparison_results"

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

if not os.path.exists(ADAPTER_ZIP):
    raise FileNotFoundError(
        f"Adapter ZIP not found: {ADAPTER_ZIP}\n"
        "Upload crm-ticket-lora.zip to this Google Drive location or update ADAPTER_ZIP."
    )

import zipfile
with zipfile.ZipFile(ADAPTER_ZIP, "r") as zf:
    zf.extractall(WORK_DIR)

# Handle ZIPs that contain an extra enclosing folder.
candidate = Path(WORK_DIR) / "crm-ticket-lora"
if not (candidate / "adapter_config.json").exists():
    configs = list(Path(WORK_DIR).rglob("adapter_config.json"))
    if not configs:
        raise FileNotFoundError(
            "Could not find adapter_config.json inside the ZIP. "
            "Check that you zipped the actual LoRA adapter folder."
        )
    candidate = configs[0].parent

ADAPTER_DIR = str(candidate)
print("Adapter directory:", ADAPTER_DIR)
print("Adapter files:", os.listdir(ADAPTER_DIR))


Mounted at /content/drive
Adapter directory: /content/crm_ai_comparison
Adapter files: ['adapter_model.safetensors', '__MACOSX', 'adapter_config.json']


## 3. Read the base model name from the adapter configuration

This avoids guessing the model repository name. We will use the exact `base_model_name_or_path` recorded when the adapter was created.

In [5]:
adapter_config_path = Path(ADAPTER_DIR) / "adapter_config.json"
with open(adapter_config_path, "r", encoding="utf-8") as f:
    adapter_config = json.load(f)

BASE_MODEL = adapter_config.get("base_model_name_or_path")
print("Base model recorded in adapter:", BASE_MODEL)

if not BASE_MODEL:
    raise ValueError(
        "adapter_config.json does not specify base_model_name_or_path. "
        "We need the exact model ID used in the training notebook."
    )


Base model recorded in adapter: unsloth/Qwen3.5-2B


## 4. Recreate the dataset and test split

The dataset card documents `dataset_clean.csv`, the ticket fields, and residual exact duplicates. We remove exact duplicate title/description pairs and rows missing essential labels, then use the previously proposed 80/10/10 split with `random_state=42` and stratification by `category_level_1`.

**Important:** if your original training notebook used different preprocessing, a different random seed, or a different split order, this is not guaranteed to reproduce its test set. For a defensible comparison, the test rows must be exactly the rows held out during training.

In [6]:
DATASET_URL = (
    "https://raw.githubusercontent.com/"
    "bazokhan/arabic-itsm-dataset/master/dataset_clean.csv"
)

df = pd.read_csv(DATASET_URL)
print("Downloaded rows:", len(df))

required = [
    "title_ar", "description_ar",
    "category_level_1", "category_level_2", "category_level_3",
    "priority", "impact", "urgency", "sentiment"
]

df = df.drop_duplicates(
    subset=["title_ar", "description_ar"],
    keep="first"
)
df = df.dropna(subset=required).reset_index(drop=True)

print("Rows after cleaning:", len(df))
print(df[["title_ar", "category_level_1", "category_level_2", "category_level_3"]].head(3))


Downloaded rows: 10000
Rows after cleaning: 9549
                     title_ar category_level_1 category_level_2  \
0                السيستم واقع          Service         Incident   
1              الأوتلوك بيهنج         Software   Email/Calendar   
2  مشكلة في الـ Remote Access          Network              VPN   

     category_level_3  
0              Outage  
1       Outlook Issue  
2  Connection Failure  


In [7]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["category_level_1"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["category_level_1"]
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

# Save a stable copy so both model runs use precisely the same test rows.
test_csv_path = f"{RESULTS_DIR}/recreated_test_set.csv"
test_df.to_csv(test_csv_path, index=False, encoding="utf-8-sig")
print("Saved test set:", test_csv_path)


Train: 7639
Validation: 955
Test: 955
Saved test set: /content/drive/MyDrive/CRM_AI/comparison_results/recreated_test_set.csv


## 5. Define the inference prompt and helper

The prompt must match the prompt used to prepare the fine-tuning examples. If you used a different `SYSTEM_PROMPT` during training, replace this one with the exact training prompt.

We deliberately request the same JSON fields used in the training examples.

In [8]:
SYSTEM_PROMPT = """
You are an IT support ticket classification assistant.

Analyze the user's support ticket and return ONLY valid JSON.

The JSON must contain:
- category
- subcategory
- issue
- priority
- impact
- urgency
- sentiment
- tags
- summary

Use the exact classification labels provided by the training examples.
Do not invent information.
Do not include markdown or explanations outside JSON.
"""

def analyze_ticket(model, processor, title, description):
    model.eval()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"Title: {title}\n\nDescription:\n{description}"
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(text=[prompt], return_tensors="pt")

    # Put model inputs on the input embedding device. This is safer than
    # assuming every model is dispatched to one device.
    input_device = model.get_input_embeddings().weight.device
    inputs = {
        k: (v.to(input_device) if hasattr(v, "to") else v)
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            use_cache=True
        )

    prompt_len = inputs["input_ids"].shape[-1]
    generated_ids = output_ids[:, prompt_len:]
    return processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0].strip()


## 6. Run the original model first

Start with a small smoke test (for example, 5 rows). If that works, increase `N_EVAL` to the full test set.

The model is deleted and GPU memory is released before loading the fine-tuned model.

In [9]:
from unsloth import FastLanguageModel

N_EVAL = 5  # Change to len(test_df) after the smoke test succeeds.
eval_df = test_df.head(N_EVAL).copy()

original_results = []
model = None
processor = None

model, processor = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=False
)
FastLanguageModel.for_inference(model)

for n, (_, row) in enumerate(eval_df.iterrows(), start=1):
    start = time.perf_counter()
    pred = analyze_ticket(
        model, processor,
        str(row["title_ar"]),
        str(row["description_ar"])
    )
    elapsed = time.perf_counter() - start
    original_results.append({
        "row_position": n - 1,
        "prediction": pred,
        "latency_seconds": elapsed
    })
    print(f"Original: {n}/{len(eval_df)}")

with open(f"{RESULTS_DIR}/original_results.json", "w", encoding="utf-8") as f:
    json.dump(original_results, f, ensure_ascii=False, indent=2)

del model, processor
model = processor = None
gc.collect()
torch.cuda.empty_cache()
print("Original run saved. GPU memory released.")
print("Free GPU memory (GB):", round(torch.cuda.mem_get_info()[0] / 1024**3, 2))


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1122 15:04:55.037000 417 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1122 15:04:55.077000 417 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.9: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

`causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
`causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


Original: 1/5
Original: 2/5
Original: 3/5
Original: 4/5
Original: 5/5
Original run saved. GPU memory released.
Free GPU memory (GB): 14.14


## 7. Run the fine-tuned model

This loads the adapter directory as the fine-tuned model using Unsloth's adapter-loading path. If the adapter was saved with a different format or Unsloth version, stop at any loading error and share it; do not modify the adapter weights manually.

In [10]:
finetuned_results = []
model = None
processor = None

model, processor = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=2048,
    load_in_4bit=False
)
FastLanguageModel.for_inference(model)

# Quick check: print the device of one LoRA parameter, if present.
for name, param in model.named_parameters():
    if "lora_" in name.lower():
        print("Example LoRA parameter:", name, param.device)
        break

for n, (_, row) in enumerate(eval_df.iterrows(), start=1):
    start = time.perf_counter()
    pred = analyze_ticket(
        model, processor,
        str(row["title_ar"]),
        str(row["description_ar"])
    )
    elapsed = time.perf_counter() - start
    finetuned_results.append({
        "row_position": n - 1,
        "prediction": pred,
        "latency_seconds": elapsed
    })
    print(f"Fine-tuned: {n}/{len(eval_df)}")

with open(f"{RESULTS_DIR}/finetuned_results.json", "w", encoding="utf-8") as f:
    json.dump(finetuned_results, f, ensure_ascii=False, indent=2)

del model, processor
model = processor = None
gc.collect()
torch.cuda.empty_cache()
print("Fine-tuned run saved. GPU memory released.")


==((====))==  Unsloth 2026.11.9: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Example LoRA parameter: base_model.model.model.language_model.layers.0.mlp.gate_proj.lora_A.default.weight cuda:0
Fine-tuned: 1/5
Fine-tuned: 2/5
Fine-tuned: 3/5
Fine-tuned: 4/5
Fine-tuned: 5/5
Fine-tuned run saved. GPU memory released.


## 8. Compare outputs and calculate metrics

This cell parses JSON predictions, compares the model's fields with the dataset labels, and reports:
- JSON validity
- Accuracy for category, subcategory, issue, and priority
- Macro F1 for the three classification levels
- Mean inference latency

The metrics are computed only on the rows actually evaluated (`N_EVAL`). After the 5-row smoke test succeeds, rerun the notebook with `N_EVAL = len(test_df)` for a meaningful evaluation.

In [11]:
from sklearn.metrics import accuracy_score, f1_score

def parse_json_prediction(text):
    if not isinstance(text, str):
        return None
    text = text.strip()
    # First try the complete response.
    try:
        obj = json.loads(text)
        return obj if isinstance(obj, dict) else None
    except Exception:
        pass

    # Fallback: extract the first JSON object from surrounding text.
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group(0))
            return obj if isinstance(obj, dict) else None
        except Exception:
            return None
    return None

with open(f"{RESULTS_DIR}/original_results.json", "r", encoding="utf-8") as f:
    original_results = json.load(f)
with open(f"{RESULTS_DIR}/finetuned_results.json", "r", encoding="utf-8") as f:
    finetuned_results = json.load(f)

def evaluate_run(results, reference_df, model_name):
    rows = []
    for i, item in enumerate(results):
        pred = parse_json_prediction(item["prediction"])
        ref = reference_df.iloc[i]
        rows.append({
            "model": model_name,
            "json_valid": pred is not None,
            "category_true": str(ref["category_level_1"]),
            "category_pred": str(pred.get("category", "")) if pred else "",
            "subcategory_true": str(ref["category_level_2"]),
            "subcategory_pred": str(pred.get("subcategory", "")) if pred else "",
            "issue_true": str(ref["category_level_3"]),
            "issue_pred": str(pred.get("issue", "")) if pred else "",
            "priority_true": str(int(ref["priority"])),
            "priority_pred": str(pred.get("priority", "")) if pred else "",
            "latency_seconds": item["latency_seconds"]
        })
    return pd.DataFrame(rows)

eval_reference = eval_df.reset_index(drop=True)
a = evaluate_run(original_results, eval_reference, "Original")
b = evaluate_run(finetuned_results, eval_reference, "Fine-tuned")

def summarize(m):
    return {
        "JSON validity": m["json_valid"].mean(),
        "Category accuracy": accuracy_score(m["category_true"], m["category_pred"]),
        "Subcategory accuracy": accuracy_score(m["subcategory_true"], m["subcategory_pred"]),
        "Issue accuracy": accuracy_score(m["issue_true"], m["issue_pred"]),
        "Priority accuracy": accuracy_score(m["priority_true"], m["priority_pred"]),
        "Category macro F1": f1_score(m["category_true"], m["category_pred"], average="macro", zero_division=0),
        "Subcategory macro F1": f1_score(m["subcategory_true"], m["subcategory_pred"], average="macro", zero_division=0),
        "Issue macro F1": f1_score(m["issue_true"], m["issue_pred"], average="macro", zero_division=0),
        "Average latency (s)": m["latency_seconds"].mean()
    }

summary = pd.DataFrame({
    "Original": summarize(a),
    "Fine-tuned": summarize(b)
})
display(summary.round(4))

summary.to_csv(f"{RESULTS_DIR}/metrics_comparison.csv")


,Original,Fine-tuned
JSON validity,1.0000,1.0000
Category accuracy,0.0000,1.0000
Subcategory accuracy,0.0000,1.0000
Issue accuracy,0.0000,0.8000
Priority accuracy,0.0000,0.2000
Category macro F1,0.0000,1.0000
Subcategory macro F1,0.0000,1.0000
Issue macro F1,0.0000,0.6667
Average latency (s),18.8311,2.4401


## 9. Inspect individual predictions

Use this table to inspect where the fine-tuned model improved or regressed. Do not treat a 5-ticket smoke test as a final result; use the full held-out test set for the final report.

In [12]:
details = eval_reference[[
    "title_ar", "description_ar",
    "category_level_1", "category_level_2", "category_level_3", "priority"
]].copy()

details["Original output"] = [x["prediction"] for x in original_results]
details["Fine-tuned output"] = [x["prediction"] for x in finetuned_results]

display(details)
details.to_csv(f"{RESULTS_DIR}/prediction_details.csv", index=False, encoding="utf-8-sig")
print("All comparison files saved in:", RESULTS_DIR)


,title_ar,description_ar,category_level_1,category_level_2,category_level_3,priority,Original output,Fine-tuned output
0,الأوتلوك مش شغال,الـ Outlook مش راضي يفتح خالص وبيقولي فيه مشكل...,Software,Email/Calendar,Outlook Issue,4,"{\n ""category"": ""Email"",\n ""subcategory"": ""O...","{""category"": ""Software"", ""subcategory"": ""Email..."
1,بج في السيستم,في مشكلة في حساب الضرايب على سيستم المبيعات. ب...,Software,Business App,Feature Request,4,"{\n ""category"": ""IT Support"",\n ""subcategory...","{""category"": ""Software"", ""subcategory"": ""Busin..."
2,Paper Jam,فيه مشكلة مرتبطة بـ Hardware / Printer/Scanner...,Hardware,Printer/Scanner,Paper Jam,2,"{\n ""category"": ""IT Support"",\n ""subcategory...","{""category"": ""Hardware"", ""subcategory"": ""Print..."
3,تحديث بيانات البروفايل,عايز اغير رقم الموبايل بتاعي في السيستم عشان ا...,Access,Account,Profile Update,2,"{\n ""category"": ""User Interface"",\n ""subcate...","{""category"": ""Access"", ""subcategory"": ""Account..."
4,مشكلة في Remote Access,الـ VPN بيقطع بعد 2-3 دقايق. محتاج أشتغل ريموت...,Network,VPN,Connection Failure,3,"{\n ""category"": ""IT Support"",\n ""subcategory...","{""category"": ""Network"", ""subcategory"": ""VPN"", ..."


All comparison files saved in: /content/drive/MyDrive/CRM_AI/comparison_results


In [13]:
%pip install -q openpyxl

In [24]:
demo_tickets = [
    {
        "id": "AR-01",
        "language": "Arabic",
        "title": "تعطل تسجيل الدخول إلى النظام",
        "description": "منذ صباح اليوم لا يستطيع موظفو قسم المبيعات تسجيل الدخول إلى نظام إدارة العملاء. تظهر رسالة كلمة المرور غير صحيحة رغم التأكد من إدخال البيانات الصحيحة. المشكلة تؤثر على جميع موظفي القسم وتمنعهم من متابعة العملاء."
    },
    {
        "id": "AR-02",
        "language": "Arabic",
        "title": "بطء شديد في الشبكة",
        "description": "الاتصال بالإنترنت بطيء جداً في الطابق الثاني منذ حوالي ساعتين. صفحات العمل تحتاج وقتاً طويلاً للتحميل والاجتماعات المرئية تتقطع. المشكلة لدى عدة موظفين وليست على جهاز واحد."
    },
    {
        "id": "AR-03",
        "language": "Arabic",
        "title": "الطابعة لا تستجيب",
        "description": "طابعة الموارد البشرية لا تطبع أي مستند، وتبقى المهام في قائمة الانتظار. أعدنا تشغيل الطابعة والحاسوب لكن المشكلة مستمرة. نحتاج طباعة عقود الموظفين قبل نهاية الدوام."
    },
    {
        "id": "AR-04",
        "language": "Arabic",
        "title": "اشتباه برسالة تصيد إلكتروني",
        "description": "وصلتني رسالة بريد إلكتروني تطلب تحديث كلمة المرور عبر رابط غير معتاد. لم أضغط على الرابط، لكن أريد التأكد من سلامة الرسالة وفحصها قبل أن يتفاعل معها أي موظف آخر."
    },
    {
        "id": "AR-05",
        "language": "Arabic",
        "title": "تعطل تطبيق المحاسبة",
        "description": "تطبيق المحاسبة يغلق مباشرة عند فتح شاشة التقارير على جهازين في قسم المالية. بدأ ذلك بعد تحديث الأمس، ولا يستطيع الفريق استخراج تقرير الإغلاق اليومي المطلوب للإدارة."
    },
    {
        "id": "EN-01",
        "language": "English",
        "title": "VPN connection keeps dropping",
        "description": "Remote employees are repeatedly disconnected from the company VPN every few minutes. They cannot access internal files or the ERP system reliably. The issue started after the network maintenance last night and affects several users."
    },
    {
        "id": "EN-02",
        "language": "English",
        "title": "New employee needs account access",
        "description": "A new finance team member starts tomorrow and needs an email account, access to the accounting application, and the shared finance folder. Please arrange the required access before 9 AM."
    },
    {
        "id": "EN-03",
        "language": "English",
        "title": "Laptop overheating and shutting down",
        "description": "My laptop becomes extremely hot during normal office work and shuts down unexpectedly two or three times a day. I have important client files and upcoming meetings, so I need help as soon as possible."
    },
    {
        "id": "EN-04",
        "language": "English",
        "title": "Possible unauthorized account access",
        "description": "I received a login notification from a country I have never visited. I did not sign in at that time. Please check whether my account has been compromised and secure it immediately."
    },
    {
        "id": "EN-05",
        "language": "English",
        "title": "Shared drive is unavailable",
        "description": "The department shared drive is not accessible to anyone in the operations team. Windows reports that the network path was not found. We have a shipment processing deadline in two hours and cannot retrieve the required documents."
    }
]

import pandas as pd

demo_df = pd.DataFrame(demo_tickets)

display(demo_df[["id", "language", "title", "description"]])

print("Total:", len(demo_df))
print("Arabic:", len(demo_df[demo_df["language"] == "Arabic"]))
print("English:", len(demo_df[demo_df["language"] == "English"]))

,id,language,title,description
0,AR-01,Arabic,تعطل تسجيل الدخول إلى النظام,منذ صباح اليوم لا يستطيع موظفو قسم المبيعات تس...
1,AR-02,Arabic,بطء شديد في الشبكة,الاتصال بالإنترنت بطيء جداً في الطابق الثاني م...
2,AR-03,Arabic,الطابعة لا تستجيب,طابعة الموارد البشرية لا تطبع أي مستند، وتبقى ...
3,AR-04,Arabic,اشتباه برسالة تصيد إلكتروني,وصلتني رسالة بريد إلكتروني تطلب تحديث كلمة الم...
4,AR-05,Arabic,تعطل تطبيق المحاسبة,تطبيق المحاسبة يغلق مباشرة عند فتح شاشة التقار...
5,EN-01,English,VPN connection keeps dropping,Remote employees are repeatedly disconnected f...
6,EN-02,English,New employee needs account access,A new finance team member starts tomorrow and ...
7,EN-03,English,Laptop overheating and shutting down,My laptop becomes extremely hot during normal ...
8,EN-04,English,Possible unauthorized account access,I received a login notification from a country...
9,EN-05,English,Shared drive is unavailable,The department shared drive is not accessible ...


Total: 10
Arabic: 5
English: 5


In [25]:
demo_tickets1 = [
    # --- Egyptian Arabic ---
    {
        "id": "EG-01",
        "language": "Arabic",
        "title": "مشكلة في تسجيل الدخول على السيستم",
        "description": "من الصبح وموظفين قسم المبيعات مش عارفين يدخلوا على سيستم إدارة العملاء. بيطلعلهم رسالة إن الباسورد غلط مع إنهم متأكدين إنهم بيكتبوا البيانات صح. المشكلة دي عند القسم كله ومعطلة شغلهم ومتابعتهم مع العملاء."
    },
    {
        "id": "EG-02",
        "language": "Arabic",
        "title": "النت بطيء جداً في الدور التاني",
        "description": "الإنترنت تقيل جداً في الدور التاني بقاله حوالي ساعتين. صفحات الشغل بتاخد وقت طويل عشان تفتح وميتنجات الأونلاين بتقطع. المشكلة عند كذا موظف مش على جهاز واحد بس."
    },
    {
        "id": "EG-03",
        "language": "Arabic",
        "title": "البرينتر مش شغالة",
        "description": "برينتر قسم الـ HR مش بتطبع أي ورق، والأوامر بتفضل متعلقة في الـ queue. عملنا ريستارت للبرينتر وللكمبيوتر وبرضه نفس المشكلة. محتاجين نطبع عقود الموظفين ضروري قبل ميعاد المرواح."
    },
    {
        "id": "EG-04",
        "language": "Arabic",
        "title": "شك في إيميل هاكينج أو نصب (Phishing)",
        "description": "وصلي إيميل بيطلب مني أغير الباسورد من لينك شكله غريب ومش مألوف. أنا مدوستش على اللينك، بس عايز أتأكد إذا كان الإيميل ده أمان ولا لأ وتفحصوه قبل ما حد تاني من الموظفين يفتحه."
    },
    {
        "id": "EG-05",
        "language": "Arabic",
        "title": "برنامج الحسابات بيقفل لوحده",
        "description": "برنامج الحسابات بيعمل كراش وبيقفل أول ما بنفتح شاشة التقارير على جهازين في قسم المالية. الموضوع ده بدأ بعد أبديت امبارح، والتيم مش عارف يطلع تقرير التقفيل اليومي المطلوب للإدارة."
    },
    {
        "id": "EG-06",
        "language": "Arabic",
        "title": "الـ VPN بيفصل كل شوية",
        "description": "الموظفين اللي شغالين من البيت الـ VPN بيفصل عندهم كل كام دقيقة. مش عارفين يدخلوا على الفايلات الداخلية ولا سيستم الـ ERP بشكل مستقر. المشكلة دي ظهرت بعد صيانة الشبكة امبارح بالليل ومأثرة على ناس كتير."
    },
    {
        "id": "EG-07",
        "language": "Arabic",
        "title": "موظف جديد محتاج صلاحيات وأكونتات",
        "description": "في موظف جديد هيبدأ بكرة في تيم المالية ومحتاج يتعمله إيميل شركة، وصلاحية دخول على برنامج الحسابات، والفولدر المشترك بتاع المالية. ياريت تخلصوا الحاجات دي قبل الساعة 9 الصبح."
    },
    {
        "id": "EG-08",
        "language": "Arabic",
        "title": "اللابتوب بيسخن وبيفصل لوحده",
        "description": "اللابتوب بتاعي بيسخن بطريقة مش طبيعية في الشغل العادي وبيفصل فجأة مرتين أو تلاتة في اليوم. عندي فايلات عملاء مهمة وميتنجات قريبة، فمحتاج مساعدة في أسرع وقت ممكن."
    },
    {
        "id": "EG-09",
        "language": "Arabic",
        "title": "حد غريب حاول يدخل على الأكونت بتاعي",
        "description": "جالي إشعار تسجيل دخول من بلد عمري ما روحتها قبل كده، وأنا مكنتش بفتح الأكونت في الوقت ده. ياريت تشيكوا إذا كان الأكونت اتهكر وتأمنوه فوراً."
    },
    {
        "id": "EG-10",
        "language": "Arabic",
        "title": "الفولدر المشترك (Shared Drive) مش شغال",
        "description": "الفولدر المشترك بتاع القسم مش راضي يفتح عند أي حد في تيم الأوبريشن. الويندوز بيطلع رسالة إن مسار الشبكة مش موجود (Network path not found). عندنا تسليم شحنة بعد ساعتين ومش عارفين نوصل للورق المطلوب."
    },

    # --- Syrian Arabic ---
    {
        "id": "SY-01",
        "language": "Arabic",
        "title": "مشكلة بتسجيل الدخول عالسيستم",
        "description": "من الصبح وموظفين قسم المبيعات مو قدرانين يفوتوا على سيستم إدارة الزباين. عم تطلع رسالة إنو كلمة السر غلط مع إنهم متأكدين عم يكتبوا البيانات صح. المشكلة صايرة مع كل القسم وموقفة شغلهم ومتابعتهم مع الزباين."
    },
    {
        "id": "SY-02",
        "language": "Arabic",
        "title": "بطء كتير كبير بالنت",
        "description": "الإنترنت كتير بطيء بالطابق التاني من شي ساعتين تقريباً. صفحات الشغل عم تاخد وقت طويل لتفتح واجتماعات الفيديو عم تقطش. المشكلة عند كذا موظف مو بس على جهاز واحد."
    },
    {
        "id": "SY-03",
        "language": "Arabic",
        "title": "الطابعة ما عم تستجيب",
        "description": "طابعة الموارد البشرية ما عم تطبع أي ملف، والأوامر عم تضل معلقة بالانتظار. عملنا إعادة تشغيل للطابعة وللكمبيوتر بس المشكلة لساتها موجودة. لازم نطبع عقود الموظفين قبل ما يخلص الدوام."
    },
    {
        "id": "SY-04",
        "language": "Arabic",
        "title": "شك بإيميل تصيد إلكتروني (Phishing)",
        "description": "وصلني إيميل عم يطلب أعمل تحديث لكلمة السر عن طريق رابط غريب مو متعودين عليه. أنا ما كبست عالرابط، بس بدي أتأكد إذا الإيميل سليم وتفحصوه قبل ما يتفاعل معه أي موظف تاني."
    },
    {
        "id": "SY-05",
        "language": "Arabic",
        "title": "برنامج المحاسبة عم يطفي لحاله",
        "description": "برنامج المحاسبة عم يسكر فوراً بس نفتح شاشة التقارير على جهازين بقسم المالية. بلشت هالمشكلة بعد تحديث مبارح، والفريق مو قدران يطلع تقرير التسكير اليومي المطلوب للإدارة."
    },
    {
        "id": "SY-06",
        "language": "Arabic",
        "title": "اتصال الـ VPN عم يفصل كل شوي",
        "description": "الموظفين اللي عم يشتغلوا عن بعد عم يفصل عندهم الـ VPN كل كم دقيقة. مو قدرانين يفوتوا عالملفات الداخلية أو سيستم الـ ERP بشكل مستقر. المشكلة بلشت بعد صيانة الشبكة مبارح بالليل ومأثرة على كذا شخص."
    },
    {
        "id": "SY-07",
        "language": "Arabic",
        "title": "موظف جديد بحاجة لحساب وصلاحيات",
        "description": "في موظف جديد رح يبلش بكرا بفريق المالية وبيحتاج إيميل، وصلاحية دخول على برنامج المحاسبة، والمجلد المشترك تبع المالية. يا ريت تجهزوا الصلاحيات المطلوبة قبل الساعة 9 الصبح."
    },
    {
        "id": "SY-08",
        "language": "Arabic",
        "title": "اللابتوب عم يحمى كتير ويطفي لحاله",
        "description": "لابتوبي عم يحمى بشكل مو طبيعي وقت الشغل العادي وعم يطفي فجأة مرتين تلاتة باليوم. عندي ملفات زباين مهمة واجتماعات قريبة، فمحتاج مساعدة بأسرع وقت ممكن."
    },
    {
        "id": "SY-09",
        "language": "Arabic",
        "title": "احتمال حدا فايت على حسابي بدون إذن",
        "description": "وصلني إشعار تسجيل دخول من بلد بحياتي ما زرته، وأنا ما فتت عالحساب بهداك الوقت. يا ريت تشيكوا إذا حسابي مخترق وتأمنوه فوراً."
    },
    {
        "id": "SY-10",
        "language": "Arabic",
        "title": "المجلد المشترك (Shared Drive) مو شغال",
        "description": "المجلد المشترك تبع القسم مو عم يفتح عند حدا بفريق العمليات. الويندوز عم يعطي رسالة إنو مسار الشبكة مو موجود. عنا تسليم شحنة بعد ساعتين ومو قدرانين نوصل للأوراق المطلوبة."
    }
]


import pandas as pd

demo_df = pd.DataFrame(demo_tickets1)

display(demo_df[["id", "language", "title", "description"]])

print("Total:", len(demo_df))
print("Arabic:", len(demo_df[demo_df["language"] == "Arabic"]))
print("English:", len(demo_df[demo_df["language"] == "English"]))

,id,language,title,description
0,EG-01,Arabic,مشكلة في تسجيل الدخول على السيستم,من الصبح وموظفين قسم المبيعات مش عارفين يدخلوا...
1,EG-02,Arabic,النت بطيء جداً في الدور التاني,الإنترنت تقيل جداً في الدور التاني بقاله حوالي...
2,EG-03,Arabic,البرينتر مش شغالة,برينتر قسم الـ HR مش بتطبع أي ورق، والأوامر بت...
3,EG-04,Arabic,شك في إيميل هاكينج أو نصب (Phishing),وصلي إيميل بيطلب مني أغير الباسورد من لينك شكل...
4,EG-05,Arabic,برنامج الحسابات بيقفل لوحده,برنامج الحسابات بيعمل كراش وبيقفل أول ما بنفتح...
5,EG-06,Arabic,الـ VPN بيفصل كل شوية,الموظفين اللي شغالين من البيت الـ VPN بيفصل عن...
6,EG-07,Arabic,موظف جديد محتاج صلاحيات وأكونتات,في موظف جديد هيبدأ بكرة في تيم المالية ومحتاج ...
7,EG-08,Arabic,اللابتوب بيسخن وبيفصل لوحده,اللابتوب بتاعي بيسخن بطريقة مش طبيعية في الشغل...
8,EG-09,Arabic,حد غريب حاول يدخل على الأكونت بتاعي,جالي إشعار تسجيل دخول من بلد عمري ما روحتها قب...
9,EG-10,Arabic,الفولدر المشترك (Shared Drive) مش شغال,الفولدر المشترك بتاع القسم مش راضي يفتح عند أي...


Total: 20
Arabic: 20
English: 0


In [26]:
import time
import json
import gc
import torch

demo_original_results = []

model = processor = None

# Load original model
model, processor = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=False
)

FastLanguageModel.for_inference(model)

# Run all 10 tickets
for i, ticket in demo_df.iterrows():

    start = time.perf_counter()

    raw_output = analyze_ticket(
        model,
        processor,
        str(ticket["title"]),
        str(ticket["description"])
    )

    elapsed = time.perf_counter() - start

    demo_original_results.append({
        "id": ticket["id"],
        "raw_json": raw_output,
        "latency_seconds": elapsed
    })

    print(
        f"Original Model: {i + 1}/10 | "
        f"{ticket['id']} | {elapsed:.2f}s"
    )

# Save raw results
with open(
    f"{RESULTS_DIR}/demo_original_results1.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        demo_original_results,
        f,
        ensure_ascii=False,
        indent=2
    )

# Release GPU memory
del model, processor

model = processor = None

gc.collect()
torch.cuda.empty_cache()

print("Original model completed. GPU memory released.")

==((====))==  Unsloth 2026.11.9: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Original Model: 1/10 | EG-01 | 4.18s
Original Model: 2/10 | EG-02 | 3.50s
Original Model: 3/10 | EG-03 | 3.80s
Original Model: 4/10 | EG-04 | 3.36s
Original Model: 5/10 | EG-05 | 4.24s
Original Model: 6/10 | EG-06 | 3.30s
Original Model: 7/10 | EG-07 | 2.99s
Original Model: 8/10 | EG-08 | 3.18s
Original Model: 9/10 | EG-09 | 3.39s
Original Model: 10/10 | EG-10 | 3.70s
Original Model: 11/10 | SY-01 | 3.62s
Original Model: 12/10 | SY-02 | 3.44s
Original Model: 13/10 | SY-03 | 3.90s
Original Model: 14/10 | SY-04 | 3.32s
Original Model: 15/10 | SY-05 | 3.90s
Original Model: 16/10 | SY-06 | 4.01s
Original Model: 17/10 | SY-07 | 3.31s
Original Model: 18/10 | SY-08 | 3.33s
Original Model: 19/10 | SY-09 | 3.38s
Original Model: 20/10 | SY-10 | 3.83s
Original model completed. GPU memory released.


In [27]:
demo_finetuned_results = []

model = processor = None

# Load fine-tuned model
model, processor = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=2048,
    load_in_4bit=False
)

FastLanguageModel.for_inference(model)

# Run the same 10 tickets
for i, ticket in demo_df.iterrows():

    start = time.perf_counter()

    raw_output = analyze_ticket(
        model,
        processor,
        str(ticket["title"]),
        str(ticket["description"])
    )

    elapsed = time.perf_counter() - start

    demo_finetuned_results.append({
        "id": ticket["id"],
        "raw_json": raw_output,
        "latency_seconds": elapsed
    })

    print(
        f"Fine-tuned Model: {i + 1}/10 | "
        f"{ticket['id']} | {elapsed:.2f}s"
    )

# Save raw results
with open(
    f"{RESULTS_DIR}/demo_finetuned_results1.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        demo_finetuned_results,
        f,
        ensure_ascii=False,
        indent=2
    )

# Release GPU memory
del model, processor

model = processor = None

gc.collect()
torch.cuda.empty_cache()

print("Fine-tuned model completed. GPU memory released.")

==((====))==  Unsloth 2026.11.9: Fast Qwen3_5 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Fine-tuned Model: 1/10 | EG-01 | 2.02s
Fine-tuned Model: 2/10 | EG-02 | 1.77s
Fine-tuned Model: 3/10 | EG-03 | 1.74s
Fine-tuned Model: 4/10 | EG-04 | 2.10s
Fine-tuned Model: 5/10 | EG-05 | 1.75s
Fine-tuned Model: 6/10 | EG-06 | 1.78s
Fine-tuned Model: 7/10 | EG-07 | 1.89s
Fine-tuned Model: 8/10 | EG-08 | 1.98s
Fine-tuned Model: 9/10 | EG-09 | 1.86s
Fine-tuned Model: 10/10 | EG-10 | 1.82s
Fine-tuned Model: 11/10 | SY-01 | 1.90s
Fine-tuned Model: 12/10 | SY-02 | 1.76s
Fine-tuned Model: 13/10 | SY-03 | 1.78s
Fine-tuned Model: 14/10 | SY-04 | 2.03s
Fine-tuned Model: 15/10 | SY-05 | 1.88s
Fine-tuned Model: 16/10 | SY-06 | 1.94s
Fine-tuned Model: 17/10 | SY-07 | 1.82s
Fine-tuned Model: 18/10 | SY-08 | 2.11s
Fine-tuned Model: 19/10 | SY-09 | 2.07s
Fine-tuned Model: 20/10 | SY-10 | 1.87s
Fine-tuned model completed. GPU memory released.


In [28]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import json
import pandas as pd

# --------------------------------------------------
# 1. JSON parser
# --------------------------------------------------

def parse_json_prediction(raw_output):

    if not raw_output:
        return None

    try:
        if isinstance(raw_output, dict):
            return raw_output

        text = str(raw_output).strip()

        # Remove markdown code fences if present
        if text.startswith("```"):
            text = text.replace("```json", "")
            text = text.replace("```", "").strip()

        return json.loads(text)

    except Exception:
        return None


# --------------------------------------------------
# 2. Match model outputs by ticket ID
# --------------------------------------------------

original_by_id = {
    item["id"]: item
    for item in demo_original_results
}

finetuned_by_id = {
    item["id"]: item
    for item in demo_finetuned_results
}


# --------------------------------------------------
# 3. Build comparison table
# --------------------------------------------------

report_rows = []

for _, ticket in demo_df.iterrows():

    original = original_by_id[ticket["id"]]
    tuned = finetuned_by_id[ticket["id"]]

    original_obj = parse_json_prediction(
        original["raw_json"]
    )

    tuned_obj = parse_json_prediction(
        tuned["raw_json"]
    )

    report_rows.append({

        "Ticket ID": ticket["id"],

        "Language": ticket["language"],

        "Title": ticket["title"],

        "Ticket Description": ticket["description"],

        "Original Model - JSON Valid":
            "Yes" if original_obj is not None else "No",

        "Original Model - Output JSON":
            original["raw_json"],

        "Original Model - Time (s)":
            round(original["latency_seconds"], 2),

        "Fine-tuned Model - JSON Valid":
            "Yes" if tuned_obj is not None else "No",

        "Fine-tuned Model - Output JSON":
            tuned["raw_json"],

        "Fine-tuned Model - Time (s)":
            round(tuned["latency_seconds"], 2)
    })


comparison_report = pd.DataFrame(report_rows)


# --------------------------------------------------
# 4. Build summary
# --------------------------------------------------

summary_report = pd.DataFrame([

    {
        "Model": "Original Qwen",

        "Tickets": len(demo_df),

        "Arabic Tickets": 5,

        "English Tickets": 5,

        "Valid JSON Responses": sum(
            parse_json_prediction(x["raw_json"]) is not None
            for x in demo_original_results
        ),

        "Average Response Time (seconds)": round(
            sum(
                x["latency_seconds"]
                for x in demo_original_results
            ) / len(demo_original_results),
            2
        )
    },

    {
        "Model": "Fine-tuned Qwen + LoRA",

        "Tickets": len(demo_df),

        "Arabic Tickets": 5,

        "English Tickets": 5,

        "Valid JSON Responses": sum(
            parse_json_prediction(x["raw_json"]) is not None
            for x in demo_finetuned_results
        ),

        "Average Response Time (seconds)": round(
            sum(
                x["latency_seconds"]
                for x in demo_finetuned_results
            ) / len(demo_finetuned_results),
            2
        )
    }

])


# --------------------------------------------------
# 5. Export Excel
# --------------------------------------------------

excel_path = (
    f"{RESULTS_DIR}/"
    "CRM_AI_10_Ticket_Demo_Comparison1.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl"
) as writer:

    summary_report.to_excel(
        writer,
        sheet_name="Quick Summary",
        index=False
    )

    comparison_report.to_excel(
        writer,
        sheet_name="Demo Comparison",
        index=False
    )


# --------------------------------------------------
# 6. Excel styling
# --------------------------------------------------

wb = load_workbook(excel_path)

header_fill = PatternFill(
    "solid",
    fgColor="17365D"
)

header_font = Font(
    color="FFFFFF",
    bold=True,
    size=11
)

thin_gray = Side(
    style="thin",
    color="D9E2F3"
)

for ws in wb.worksheets:

    ws.freeze_panes = "A2"

    ws.auto_filter.ref = ws.dimensions

    ws.sheet_view.showGridLines = False

    for cell in ws[1]:

        cell.fill = header_fill

        cell.font = header_font

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

        cell.border = Border(
            bottom=thin_gray
        )

    ws.row_dimensions[1].height = 32


# Detailed comparison sheet
ws = wb["Demo Comparison"]

widths = {
    "A": 12,
    "B": 13,
    "C": 34,
    "D": 58,
    "E": 19,
    "F": 62,
    "G": 17,
    "H": 20,
    "I": 62,
    "J": 17
}

for col, width in widths.items():
    ws.column_dimensions[col].width = width


for row in ws.iter_rows(min_row=2):

    ws.row_dimensions[row[0].row].height = 150

    for cell in row:

        cell.alignment = Alignment(
            vertical="top",
            wrap_text=True
        )

    # JSON formatting
    row[5].font = Font(
        name="Consolas",
        size=9
    )

    row[8].font = Font(
        name="Consolas",
        size=9
    )


# Summary sheet
summary_ws = wb["Quick Summary"]

summary_widths = {
    "A": 30,
    "B": 14,
    "C": 18,
    "D": 18,
    "E": 24,
    "F": 34
}

for col, width in summary_widths.items():
    summary_ws.column_dimensions[col].width = width

for row in summary_ws.iter_rows(min_row=2):

    for cell in row:

        cell.alignment = Alignment(
            vertical="center",
            wrap_text=True
        )

    summary_ws.row_dimensions[row[0].row].height = 30


wb.save(excel_path)


# --------------------------------------------------
# 7. Display results
# --------------------------------------------------

print("Excel report created successfully!")
print(excel_path)

display(summary_report)

display(
    comparison_report[
        [
            "Ticket ID",
            "Language",
            "Title",
            "Original Model - JSON Valid",
            "Fine-tuned Model - JSON Valid"
        ]
    ]
)

Excel report created successfully!
/content/drive/MyDrive/CRM_AI/comparison_results/CRM_AI_10_Ticket_Demo_Comparison1.xlsx


,Model,Tickets,Arabic Tickets,English Tickets,Valid JSON Responses,Average Response Time (seconds)
0,Original Qwen,20,5,5,20,3.58
1,Fine-tuned Qwen + LoRA,20,5,5,20,1.89


,Ticket ID,Language,Title,Original Model - JSON Valid,Fine-tuned Model - JSON Valid
0,EG-01,Arabic,مشكلة في تسجيل الدخول على السيستم,Yes,Yes
1,EG-02,Arabic,النت بطيء جداً في الدور التاني,Yes,Yes
2,EG-03,Arabic,البرينتر مش شغالة,Yes,Yes
3,EG-04,Arabic,شك في إيميل هاكينج أو نصب (Phishing),Yes,Yes
4,EG-05,Arabic,برنامج الحسابات بيقفل لوحده,Yes,Yes
5,EG-06,Arabic,الـ VPN بيفصل كل شوية,Yes,Yes
6,EG-07,Arabic,موظف جديد محتاج صلاحيات وأكونتات,Yes,Yes
7,EG-08,Arabic,اللابتوب بيسخن وبيفصل لوحده,Yes,Yes
8,EG-09,Arabic,حد غريب حاول يدخل على الأكونت بتاعي,Yes,Yes
9,EG-10,Arabic,الفولدر المشترك (Shared Drive) مش شغال,Yes,Yes
